# Inspect, train, evaluate and export
Upload the reviewed trainer, experiment.json and data files to this runtime. Choose the actual runtime in Colab, then inspect hardware below. No install, account access or cloud connection is inferred. Prepare a locked transformer environment separately.
Run cells in order. Each experiment uses new output directories; change the paths to start a separate run. Do not overwrite checkpoints. Keep runtime files in durable storage before disconnecting.


In [ ]:
from pathlib import Path
import json, sys, subprocess
SPEC = Path("experiment.json").resolve()
spec = json.loads(SPEC.read_text())
TRAINER = Path("ml-transformer-train.py" if spec["recipe"] in ("embedding", "lora") else "ml-lab.py").resolve()
SMOKE = Path("runs/smoke").resolve()
RUN = Path("runs/full").resolve()
assert TRAINER.is_file() and SPEC.is_file()


In [ ]:
import platform, importlib.util
print({"python":sys.version.split()[0],"platform":platform.platform(),"torchInstalled":importlib.util.find_spec("torch") is not None})
if importlib.util.find_spec("torch"):
 import torch
 print({"cudaAvailable":torch.cuda.is_available(),"device":torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})


In [ ]:
validation = subprocess.run([sys.executable,"-I",str(TRAINER),"--spec",str(SPEC),"--check"],check=True,capture_output=True,text=True)
print(validation.stdout)


In [ ]:
assert not SMOKE.exists(), "Inspect the existing smoke receipt or choose a new directory before rerunning"
subprocess.run([sys.executable,"-I",str(TRAINER),"--spec",str(SPEC),"--output",str(SMOKE),"--steps","2"],check=True)
smoke = json.loads((SMOKE / "metrics.json").read_text())
assert (SMOKE / "checkpoint.json").is_file() and smoke["status"] == "executed"
print(smoke)


In [ ]:
assert not RUN.exists(), "Inspect the existing run and checkpoints or choose a new directory before rerunning"
subprocess.run([sys.executable,"-I",str(TRAINER),"--spec",str(SPEC),"--output",str(RUN),"--resume",str(SMOKE / "checkpoint.json")],check=True)


In [ ]:
metrics = json.loads((RUN / "metrics.json").read_text())
assert metrics["status"] == "executed"
print(json.dumps(metrics,indent=2))
# Inspect frozen test slices, baseline, uncertainty and task-specific failures.
# Offline Q residuals and causal loss alone do not establish policy/task success.


In [ ]:
import hashlib
artifacts = [p for p in RUN.rglob("*") if p.is_file() and (p.name in ("metrics.json","model.json","checkpoint.json","manifest.json") or "export" in p.parts)]
assert artifacts
# Copy these artifacts and complete Trainer checkpoints to chosen durable storage.
# Validate the exported model on the real target before deployment.
def file_hash(file):
 h = hashlib.sha256()
 with file.open("rb") as handle:
  for chunk in iter(lambda:handle.read(1048576),b""): h.update(chunk)
 return h.hexdigest()
print([{ "path":str(p),"bytes":p.stat().st_size,"sha256":file_hash(p)} for p in artifacts])
